In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torchvision
import numpy as np

plt.rcParams['figure.figsize'] = (16, 8)

In [2]:
print("Pytorch version: ", torch.__version__)
if torch.cuda.is_available():
    # Prints the version that PyTorch thinks it should be using
    print(f"Torch compiled for CUDA version: {torch.version.cuda}")
    print(f"System can use GPU: {torch.cuda.get_device_name(0)}")
else:
    print("PyTorch cannot detect CUDA. The setup is broken.")

Pytorch version:  2.12.0+cu126
Torch compiled for CUDA version: 12.6
System can use GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [4]:
transform_normalize = torchvision.transforms.Compose([
    torchvision.transforms.RandomCrop(32, padding=4),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

train_dataset = torchvision.datasets.CIFAR10(root ='./CIFAR10',train=True,transform=transform_normalize,download=True)
test_dataset = torchvision.datasets.CIFAR10(root ='./CIFAR10',train=False,transform=transform_normalize,download=True)

KeyboardInterrupt: 

In [13]:
batch_size = 128
train_loader = torch.utils.data.DataLoader(dataset = train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset = test_dataset, batch_size=batch_size, shuffle=True)

In [14]:
dataiter = iter(train_loader)
images,labels = next(dataiter)

plt.imshow(np.transpose(torchvision.utils.make_grid(images[:25],normalize=True,nrow=5,padding=1).numpy()))
plt.axis('off')
plt.show()

RuntimeError: output with shape [1, 32, 32] doesn't match the broadcast shape [3, 32, 32]

In [93]:
print(f"Train size: {len(train_loader)}")  # Should print 50000
print(f"Test size: {len(test_loader)}")    # Should print 10000

Train size: 391
Test size: 79


In [102]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3,stride=1,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,stride=1,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(in_channels=64,out_channels=128,kernel_size=3,stride=1,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Flatten(),
            nn.Linear(128 *4*4,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )

    def forward(self,x):
        return self.model(x)

In [106]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print('CUDA available')
else:
    print('CPU available')
model = CNN().to(device)

num_epochs = 50
learning_rate = 0.001
weight_decay = 0.01
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=learning_rate, weight_decay=weight_decay
)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,num_epochs)

CUDA available


In [111]:
train_loss_list = []
for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}:', end=' ')
    train_loss = 0
    train_total ,correct= 0,0
    train_acc,test_acc=[],[]
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model.forward(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()*images.size(0)
        train_total += labels.size(0)
        _,predicted=outputs.max(1)
        correct+=predicted.eq(labels).sum().item()
    running_loss = train_loss/train_total
    train_loss_list.append(running_loss)
    train_acc = (correct/train_total)*100

    model.eval()
    test_total,test_correct = 0,0
    with torch.no_grad():
        for images, labels in test_loader:
            images,labels = images.to(device),labels.to(device)
            outputs = model.forward(images)
            test_total += labels.size(0)
            _,predicted=outputs.max(1)
            test_correct += predicted.eq(labels).sum().item()
        test_acc = (test_correct/test_total)*100

    print(f"Training loss = {train_loss_list[-1]:.4f}| Training acc = {train_acc:.4f}%| Test acc = {test_acc:.4f}%")
    scheduler.step()

Epoch 1/50: Training loss = 1.0738| Training acc = 62.3560%| Test acc = 64.7900%
Epoch 2/50: Training loss = 1.0495| Training acc = 62.9180%| Test acc = 62.3000%
Epoch 3/50: Training loss = 1.0159| Training acc = 64.2020%| Test acc = 63.7600%
Epoch 4/50: Training loss = 1.0076| Training acc = 64.6860%| Test acc = 63.8100%
Epoch 5/50: Training loss = 0.9960| Training acc = 65.1240%| Test acc = 66.4600%
Epoch 6/50: Training loss = 0.9755| Training acc = 65.8420%| Test acc = 65.6100%
Epoch 7/50: Training loss = 0.9644| Training acc = 66.2420%| Test acc = 67.2400%
Epoch 8/50: Training loss = 0.9605| Training acc = 66.3940%| Test acc = 67.9800%
Epoch 9/50: Training loss = 0.9509| Training acc = 66.6780%| Test acc = 67.6300%
Epoch 10/50: Training loss = 0.9398| Training acc = 67.2880%| Test acc = 66.4700%
Epoch 11/50: Training loss = 0.9318| Training acc = 67.3580%| Test acc = 67.3600%
Epoch 12/50: Training loss = 0.9289| Training acc = 67.7280%| Test acc = 68.3300%
Epoch 13/50: Training los

In [112]:
import torch.nn.functional as F
import torch.nn as nn
class Residual(nn.Module):
    def __init__(self, in_channel, out_channel, is_conv1=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel,out_channels=out_channel, kernel_size=3, stride=strides, padding=1)
        self.conv2 = nn.Conv2d(in_channels=out_channel,out_channels=out_channel, kernel_size=3, stride=1, padding=1)
        if is_conv1:
            self.conv3=nn.Conv2d(in_channels=in_channel,out_channels=out_channel, kernel_size=1, stride=strides, padding=0)
        else:
            self.conv3=None
        self.b1 = nn.BatchNorm2d(out_channel)
        self.b2 = nn.BatchNorm2d(out_channel)

    def forward(self,X):
      Y = F.relu(self.b1(self.conv1(X)))
      Y = self.b2(self.conv2(Y))
      if self.conv3 is not None:
          X = self.conv3(X)
      Y += X
      return F.relu(Y)


In [114]:
class ResNet(nn.Module):

    def __init__(self, arch, lr=0.1, num_classes=10):
        super().__init__()
        self.resnet = nn.Sequential(self.b1())

        for i,(num_blocks, in_channel,out_channel) in enumerate(arch):
            self.resnet.add_module(f'block{i+2}',
                                   self.block(num_blocks,in_channel,out_channel, first_block=(num_blocks==0)))

        self.resnet.add_module('last', nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(arch[-1][2], num_classes)
        ))
        self.resnet.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            nn.init.zeros_(m.bias)

    def b1(self):
        return nn.Sequential(
            nn.Conv2d(3,64, kernel_size=7, stride=2, padding=3),
            nn.LazyBatchNorm2d(), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

    def block(self, num_block, in_channel,out_channel, first_block=False):
        blck=[]
        for i in range(num_block):
            if i==0 and not first_block:
                blck.append(Residual(in_channel, out_channel, is_conv1=True, strides=2))
            else:
                blck.append(Residual(out_channel,out_channel))
        return nn.Sequential(*blck)
    def forward(self, x):
        return self.resnet(x)

In [115]:
class ResNet18(ResNet):
    def __init__(self,lr=0.1,num_classes=10):
        super().__init__(
            ((2,64,128),(2,128,256),(2,256,512)),
            lr,
            num_classes
        )


In [116]:
model = ResNet18()
x = torch.randn(2, 3, 32, 32)
out = model(x)
print(out.shape)

torch.Size([2, 10])


In [117]:
loss_list,acc_list=[],[]
epochs=50
criterion = torch.nn.CrossEntropyLoss()
model=ResNet18(lr=0.1,num_classes=10)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

model.to(device)
for e in range(epochs):
    print(f'Epoch {e+1}/{epochs}:', end=' ')
    running_loss = 0.0
    train_correct ,train_total= 0,0
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _,predicted=outputs.max(1)
        train_total+= labels.size(0)
        train_correct+= predicted.eq(labels).sum().item()

    training_loss = running_loss/train_total
    train_acc = 100*train_correct/train_total
    loss_list.append(training_loss)
    acc_list.append(train_acc)

    model.eval()
    correct,total=0,0
    test_acc =0
    test_list=[]
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _,predicted=outputs.max(1)
            total+= labels.size(0)
            correct+= predicted.eq(labels).sum().item()
        test_acc = 100*correct/total
        test_list.append(test_acc)
    print(f'Training loss = {training_loss:.4f}| Training acc = {train_acc:.4f}%| Test acc = {test_acc:.4f}%')
    scheduler.step()

Epoch 1/50: Training loss = 2.8770| Training acc = 16.9320%| Test acc = 27.3200%
Epoch 2/50: Training loss = 1.7552| Training acc = 33.4480%| Test acc = 37.9600%
Epoch 3/50: Training loss = 1.5602| Training acc = 41.7160%| Test acc = 44.4600%
Epoch 4/50: Training loss = 1.4476| Training acc = 46.9000%| Test acc = 45.9500%
Epoch 5/50: Training loss = 1.3531| Training acc = 51.0580%| Test acc = 52.2600%
Epoch 6/50: Training loss = 1.2540| Training acc = 55.0920%| Test acc = 50.7500%
Epoch 7/50: Training loss = 1.1834| Training acc = 58.1240%| Test acc = 58.0200%
Epoch 8/50: Training loss = 1.1142| Training acc = 60.7820%| Test acc = 54.9200%
Epoch 9/50: Training loss = 1.0650| Training acc = 62.9380%| Test acc = 60.8900%
Epoch 10/50: Training loss = 1.0231| Training acc = 64.3320%| Test acc = 61.0800%
Epoch 11/50: Training loss = 0.9968| Training acc = 65.5040%| Test acc = 56.7700%
Epoch 12/50: Training loss = 0.9622| Training acc = 66.5300%| Test acc = 62.2000%
Epoch 13/50: Training los